# Generate synthetic bioprocess data

This notebook introduces the Milestone 2 dataset before it is used by a database or model. The equations are educational approximations, and every microscopy-style image is synthetic.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import plotly.express as px

from src.data.generate_data import SimulationConfig, write_dataset

## Generate the complete reproducible dataset

The fixed seed makes repeated runs identical. Fifty batches sampled from 0 through 240 hours every four hours produce 3,050 sensor rows.

In [ ]:
config = SimulationConfig(n_batches=50, random_seed=42)
dataset = write_dataset(project_root, config)

{
    "batches": len(dataset.batches),
    "sensor_rows": len(dataset.sensor_data),
    "notes": len(dataset.text_records),
    "images": len(dataset.images),
}

## Inspect metadata, measurements, and planted scenarios

Scenario truth is kept in a separate evaluation table so it cannot accidentally become a model feature.

In [ ]:
display(dataset.batches.head())
display(dataset.sensor_data.head())
display(dataset.scenario_truth.query("is_abnormal"))

## Compare a normal run with the DO-instability run

B014 contains controlled DO oscillations from 72 to 96 hours. Compare it with B001 before moving to database work.

In [ ]:
comparison = dataset.sensor_data.query("batch_id in ['B001', 'B014']")
fig = px.line(
    comparison,
    x="time_hr",
    y="do_pct",
    color="batch_id",
    labels={"time_hr": "Time (h)", "do_pct": "DO (% air saturation)"},
    title="Dissolved oxygen: normal and planted-instability runs",
)
fig.show()

## Inspect the main trajectories

These small multiples show cell growth, substrate, metabolite, and product behavior for selected batches.

In [ ]:
variables = [
    "viable_cell_density_million_ml",
    "glucose_g_l",
    "lactate_g_l",
    "titer_g_l",
]
selected = dataset.sensor_data.query("batch_id in ['B001', 'B007', 'B023', 'B044']")
long_data = selected.melt(
    id_vars=["batch_id", "time_hr"],
    value_vars=variables,
    var_name="measurement",
    value_name="value",
)
fig = px.line(
    long_data,
    x="time_hr",
    y="value",
    color="batch_id",
    facet_row="measurement",
    height=900,
)
fig.update_yaxes(matches=None)
fig.show()

## Files written

The CSV files are under `data/raw/`, scenario truth is under `data/evaluation/`, and illustrative PNG files are under `images/`. `data/raw/manifest.json` records configuration, units, assumptions, row counts, and file hashes.